In [6]:
from fastai.torch_basics import *
from fastai.data.all import *
from fastai.text.data import *
from bisect import bisect_right
from tokenizers import Tokenizer
from datasets import load_dataset, get_dataset_split_names
import pyarrow as pa


/Users/hoa/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [7]:
#| default_exp text.tokenizer


# Text tokenizer

> Tokenize texts with Hugging Face tokenizers into memory-mapped Arrow caches and build `DataLoaders`

This module encodes text with pretrained Hugging Face tokenizers, stores variable-length token sequences in memory-mapped Arrow caches alongside document metadata, and builds fastai `DataLoaders` from those caches. `hf_tokenizer` loads a pretrained tokenizer and adds `EOS_TOK`, `BOS_TOK` and `PAD_TOK`.

In [8]:
EOS_TOK = "<eos>"
BOS_TOK = "<bos>"
PAD_TOK = "<pad>"

def hf_tokenizer(name='gpt2'):
    "Pretrained Hugging Face tokenizer `name`, with `EOS_TOK`, `BOS_TOK` and `PAD_TOK` added as special tokens"
    tok = Tokenizer.from_pretrained(name)
    tok.add_special_tokens([EOS_TOK, BOS_TOK, PAD_TOK])
    return tok

In [9]:
tok = hf_tokenizer()
pad_id = tok.token_to_id(PAD_TOK)
for t in (EOS_TOK, BOS_TOK, PAD_TOK): test_ne(tok.token_to_id(t), None)
tok.get_vocab_size(), pad_id

(50260, 50259)

## Encoding texts

Documents have different lengths, so they don't fit in a single rectangular array. `tokenize_texts` returns an Arrow list column with one list of token IDs per document. Under the hood, Arrow stores a flat array of all token IDs alongside an offsets array marking document boundaries, giving us fast, compact storage and direct per-document access without managing those two arrays by hand.

In [10]:
def id_dtype(tok):
    "The smallest type that holds every ID of `tok`"
    return np.uint16 if tok.get_vocab_size(with_added_tokens=True) <= 2**16 else np.uint32

In [11]:
def tokenize_batches(
    items, # Texts, or the items `get_texts` extracts them from
    tok, # HF `Tokenizer`
    get_texts=None, # Column of texts, or function from a batch of `items` to its texts
    bs=1024, # Texts per tokenizer batch
):
    "Encode each `bs` texts into an Arrow list array, where document `i` is `values[offsets[i]:offsets[i+1]]`"
    if isinstance(get_texts, str): get_texts = itemgetter(get_texts)
    dtype = id_dtype(tok)
    for i in progress_bar(range(0, len(items), bs)):
        batch = items[i:i+bs]
        ids = [np.asarray(e.ids, dtype=dtype) for e in tok.encode_batch_fast(list(batch if get_texts is None else get_texts(batch)))]
        yield pa.LargeListArray.from_arrays(np.cumsum([0] + [len(o) for o in ids]), np.concatenate(ids))

@delegates(tokenize_batches)
def tokenize_texts(items, tok, **kwargs):
    "Encode texts into an Arrow column with one list of token IDs per document"
    return pa.chunked_array(list(tokenize_batches(items, tok, **kwargs)), type=pa.large_list(pa.from_numpy_dtype(id_dtype(tok))))


In [12]:
texts = ["The quick brown fox", "jumps over the lazy dog"]
ids = tokenize_texts(texts, tok)
test_eq(len(ids), len(texts))
ids[0]

 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

<pyarrow.LargeListScalar: [464, 2068, 7586, 21831]>

Each row is one document. Decode it with the same tokenizer.


In [13]:
tok.decode(ids[0].as_py())


'The quick brown fox'

For tabular input, `get_texts` names the text column. 


In [14]:
df = pd.DataFrame({'id': range(len(texts)), 'text': texts})
test_eq(tokenize_texts(df, tok, get_texts='text').to_pylist(), ids.to_pylist())


 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

`TokenDocs` wraps the column. It reads each chunk's flat token array and offsets once, as NumPy views that copy nothing, and keeps an `idxs` array of which documents this view sees. Indexing with a position returns one document as a NumPy slice. Indexing with a list or slice (such as fastai creating train and validation splits) returns a lightweight subset view with updated `idxs` that shares the same arrays. The `iloc` property lets fastai index this collection directly without converting it to a list.

In [15]:
class TokenDocs:
    "Document views backed by an Arrow list column of token IDs"
    def __init__(
        self,
        ids # Arrow list array or chunked array with one list of token IDs per document
    ):
        self.ids = ids
        chunks = ids.chunks if isinstance(ids, pa.ChunkedArray) else [ids]
        self.chunks = [(c.values.to_numpy(), c.offsets.to_numpy()) for c in chunks]
        self.starts = np.cumsum([0] + [len(c) for c in chunks]).tolist()
        self.lens = np.concatenate([np.diff(o) for _,o in self.chunks]) if chunks else np.empty(0, dtype=np.int64)
        self.idxs = np.arange(self.starts[-1])

    def __len__(self): return len(self.idxs)
    def __repr__(self): return f'TokenDocs({len(self)} documents)'
    @property
    def iloc(self): return self

    def __getitem__(self, i):
        i = self.idxs[i]
        if not is_indexer(i): # subset view (e.g. train/val splits) sharing the same token arrays
            res = copy(self)
            res.idxs = i
            return res
        c = bisect_right(self.starts, i)-1
        flat, offsets = self.chunks[c]
        i -= self.starts[c]
        return flat[offsets[i]:offsets[i+1]]

In [16]:
docs = TokenDocs(ids)
test_eq([tok.decode(d.tolist()) for d in docs], texts)
test_eq(docs[[1, 0]][0], docs[-1])
test_is(docs[[1, 0]].chunks, docs.chunks)
test_eq(np.shares_memory(docs[0], ids.chunk(0).values.to_numpy()), True)
tok.decode(docs[1].tolist())


'jumps over the lazy dog'

A column can have several chunks, for example one per tokenizer batch or one per cache file. `TokenDocs` finds the chunk that holds each document.

In [17]:
chunked = TokenDocs(tokenize_texts(texts, tok, bs=1))
test_eq(len(chunked.chunks), 2)
test_eq(list(chunked), list(docs))

 |███████████████████████████████| 100.00% [2/2 00:00<00:00]

## Token cache

A cache is one Arrow file. Its `ids` column holds the token IDs. Its other columns, called `xtra`, hold one value per document, such as a label, a file path or a source split. Each row keeps a document together with its `xtra` values. The tokenizer is saved in the file's schema. Raw text is not stored.

`save_token_cache` writes one record batch per array of `ids`. `load_token_cache` memory-maps the file, and the documents it returns are views of the file. Given several files, it returns their documents as one collection.

In [18]:
def save_token_cache(
    path, # Arrow file to write
    ids, # Arrow list arrays of token IDs, such as the batches from `tokenize_batches`
    tok, # Tokenizer that produced `ids`
    xtra, # DataFrame of per-document columns (e.g. labels, file paths, splits)
):
    "Save token IDs, `xtra` columns and the tokenizer to the Arrow file `path`"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    meta = pa.Table.from_pandas(xtra, preserve_index=False).combine_chunks()
    schema = pa.schema([pa.field('ids', pa.large_list(pa.from_numpy_dtype(id_dtype(tok)))), *meta.schema],
        metadata=dict(tokenizer=tok.to_str()))
    tmp, n = path.with_suffix('.tmp'), 0
    try:
        with pa.ipc.new_file(tmp, schema) as w:
            for a in (ids.chunks if isinstance(ids, pa.ChunkedArray) else ids):
                rows = meta.slice(n, len(a))
                if len(rows) != len(a): raise ValueError('xtra must have one row per document')
                w.write_batch(pa.RecordBatch.from_arrays([a, *(c.chunk(0) for c in rows.columns)], schema=schema))
                n += len(a)
        if n != len(xtra): raise ValueError('xtra must have one row per document')
        tmp.replace(path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise

In [19]:
def load_token_cache(path):
    "Memory-map one or more Arrow caches, returning their documents, tokenizer and `xtra` columns"
    tbl = pa.concat_tables([pa.ipc.open_file(pa.memory_map(str(p))).read_all() for p in L(path)], promote_options='default')
    tok = Tokenizer.from_str(tbl.schema.metadata[b'tokenizer'].decode())
    return TokenDocs(tbl['ids']), tok, tbl.drop_columns('ids').to_pandas()

In [20]:
with tempfile.TemporaryDirectory() as d:
    cache = Path(d)/'docs.arrow'
    xtra = pd.DataFrame(dict(path=['train/pos/1.txt', 'test/neg/2.txt'], label=['pos', 'neg'], split=['train', 'test']))
    save_token_cache(cache, ids, tok, xtra)
    docs2, tok2, xtra2 = load_token_cache(cache)
    test_eq(list(docs2), list(docs))
    test_eq(tok2.to_str(), tok.to_str())
    test_eq(xtra2.to_dict('list'), xtra.to_dict('list'))
xtra2

,path,label,split
0,train/pos/1.txt,pos,train
1,test/neg/2.txt,neg,test


`load_token_cache` combines several cache files into one `TokenDocs` collection. `save_token_cache` rejects an `xtra` table whose row count does not match the documents.

In [21]:
with tempfile.TemporaryDirectory() as d:
    c1, c2 = Path(d)/'a.arrow', Path(d)/'b.arrow'
    save_token_cache(c1, ids[:1], tok, pd.DataFrame(dict(split=['a'])))
    save_token_cache(c2, ids[1:], tok, pd.DataFrame(dict(split=['b'])))
    both_docs, _, both_xtra = load_token_cache([c1, c2])
    test_eq(list(both_docs), list(docs))
    test_eq(both_xtra.split.tolist(), ['a', 'b'])
    with expect_fail(ValueError, contains='one row per document'):
        save_token_cache(Path(d)/'bad.arrow', ids, tok, pd.DataFrame(dict(split=['only_one'])))
both_xtra

,split
0,a
1,b


## Hugging Face datasets

We’ll prepare datasets for training by tokenizing their documents once and preserving the matching labels and source splits. The resulting cache can supply both language-model and classification loaders. Each source split has its own cache file, so splits can be combined in any way without tokenizing them again.

We’ll start with local CSV files and SimpleWiki, where each row is one document. WikiText shows how to prepare data when an article spans several rows.


In [22]:
def get_dataset_data(
    name, # Hugging Face Hub dataset name or local dataset directory
    split, # Split name or expression, such as `'train'` or `'train[:1%]'`
    get_data=None, # Optional callable returning (texts, xtra DataFrame) for the split
    subset=None, # Dataset configuration name
    text_col=None, # Text column; required when get_data is not supplied
    xtra_cols=(), # Columns to cache alongside the source split
):
    "Load one dataset split and prepare its texts and `xtra` columns"
    if get_data is None and text_col is None: raise ValueError('Provide text_col or get_data')
    ds = load_dataset(name, subset, split=split)
    if get_data is None: texts, xtra = ds[text_col], pd.DataFrame({c: ds[c] for c in xtra_cols}, index=range(len(ds)))
    else: texts, xtra = get_data(ds)
    return texts, xtra.assign(split=split)


These local CSVs contain one review per row. Loading them uses the same column-based interface as a Hub dataset, and each `xtra` row records its source split.

In [23]:
from contextlib import contextmanager

In [24]:
@contextmanager
def sample_dataset():
    with tempfile.TemporaryDirectory() as d:
        source = Path(d)/'source'
        source.mkdir()
        for split_name in ['train', 'validation', 'test']:
            (source/f'{split_name}.csv').write_text(rf"""text,label
Great {split_name} film,pos
Awful {split_name} film,neg
""")
        yield str(source), Path(d)/'cache'

`get_dataset_data` loads one split. It returns the texts and `xtra`, with one row per text holding the source split and the columns named in `xtra_cols`.


In [25]:
with sample_dataset() as (source, _):
    train_texts, train_xtra = get_dataset_data(source, 'train', text_col='text', xtra_cols=['label'])
    test_eq(list(train_texts), ['Great train film', 'Awful train film'])
    test_eq(train_xtra.split.tolist(), ['train', 'train'])
train_xtra


Generating train split: 2 examples [00:00, 2114.06 examples/s]
Generating validation split: 2 examples [00:00, 3775.25 examples/s]
Generating test split: 2 examples [00:00, 4328.49 examples/s]


,label,split
0,pos,train
1,neg,train


Supply either a text column or a custom data callable when preparing a dataset.

In [26]:
with expect_fail(ValueError, contains='Provide text_col or get_data'): get_dataset_data('stanfordnlp/imdb', 'train')


In [27]:
@delegates(get_dataset_data)
def tokenize_dataset(
    name, # Hugging Face Hub dataset name or local dataset directory
    subset=None, # Dataset configuration name
    split=None, # All splits if None, or a split name/expression or list of them
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Cache directory; defaults to a path keyed by source and tokenizer
    bs=1024, # Texts per tokenizer batch
    **kwargs
):
    "Build or load one Arrow token cache per split of a Hugging Face Hub dataset or local dataset directory"
    cache_dir = Path(cache_dir or fastai_path('data')/'token_cache'/str(name).lstrip('/')/(subset or 'default')/tok)
    if split is None: split = get_dataset_split_names(name, subset)
    paths = [cache_dir/f'{s}.arrow' for s in L(split)]
    tokenizer = None
    for s,path in zip(L(split), paths):
        if path.exists(): continue
        texts, xtra = get_dataset_data(name, s, subset=subset, **kwargs)
        if tokenizer is None: tokenizer = hf_tokenizer(tok)
        save_token_cache(path, tokenize_batches(texts, tokenizer, bs=bs), tokenizer, xtra)
    return load_token_cache(paths)

`tokenize_dataset` caches each split in its own file and returns the documents of all the requested splits together. `split=None` reads every split of the dataset.


In [28]:
with sample_dataset() as (source, cache_dir):
    review_docs, _, review_xtra = tokenize_dataset(source, text_col='text', xtra_cols=['label'], split=['train', 'validation'], cache_dir=cache_dir)
    test_eq(sorted(f.name for f in cache_dir.ls()), ['train.arrow', 'validation.arrow'])
    test_eq(len(review_docs), 4)
    test_eq(review_xtra.split.tolist(), ['train', 'train', 'validation', 'validation'])
    test_eq(review_xtra.label.tolist(), ['pos', 'neg']*2)
review_xtra


Generating train split: 2 examples [00:00, 3125.41 examples/s]
Generating validation split: 2 examples [00:00, 4236.67 examples/s]
Generating test split: 2 examples [00:00, 4660.34 examples/s]


 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

,label,split
0,pos,train
1,neg,train
2,pos,validation
3,neg,validation


Simple English Wikipedia has one article per row in its `md` column. Pass `text_col='md'` to tokenize those articles. `tokenize_dataset` adds each article's source split to `xtra`.

In [29]:
#| eval: false
docs_sw, _, xtra_sw = tokenize_dataset('answerdotai/simplewiki', subset='articles', text_col='md', split='train[:500]')
len(docs_sw), docs_sw.lens.sum()


(500, np.int64(770199))

In [30]:
#| eval: false
tok.decode(docs_sw[0].tolist())[:80]


'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian cal'

WikiText-103 spreads each article across several rows, with headings marking article boundaries. Here a custom `get_data` callable joins those rows and returns `(texts, xtra)`, with one `xtra` row per assembled article. It runs separately on each source split to keep articles within their split.

In [31]:
def get_wiki103_data(ds):
    "Assemble WikiText-103 articles from source lines"
    heading, docs, cur = re.compile(r'^ = [^=].* = \n$'), [], []
    for text in ds['text']:
        if heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    texts = [d for d in (*docs, ''.join(cur)) if d.strip()]
    return texts, pd.DataFrame(index=range(len(texts)))

In [32]:
#| eval: false
docs_wiki, tok_wiki, xtra_wiki = tokenize_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1', get_data=get_wiki103_data, split='validation')
len(docs_wiki), docs_wiki.lens.sum()


(60, np.int64(247289))

Only the requested validation split is written to disk, and decoding the first document confirms the article boundary join.

In [33]:
#| eval: false
sorted(p.name for p in (fastai_path('data')/'token_cache/Salesforce/wikitext/wikitext-103-raw-v1/gpt2').iterdir())


['train[:1%].arrow', 'validation.arrow']

In [34]:
#| eval: false
tok_wiki.decode(docs_wiki[0].tolist())[:80]


' = Homarus gammarus = \n Homarus gammarus , known as the European lobster or comm'

## Text files

For the standard `split/label/file` layout, directory names supply the label and source split. The relative path identifies the document. `get_folder_data` sorts the files and keeps `xtra` in that order.

For other layouts, `get_xtra` receives the sorted relative paths and returns more `xtra` columns with one row per file. The paths are added automatically.

In [35]:
def get_folder_data(
    path, # Folder of text files
    get_xtra=None # Callable returning `xtra` columns for nonstandard folder layouts
):
    "List sorted text files under `path` and prepare their `xtra` columns"
    path = Path(path)
    files = get_text_files(path).sorted()
    rel_files = [f.relative_to(path) for f in files]
    paths = [str(f) for f in rel_files]
    if get_xtra is None:
        labels = [f.parent.name if len(f.parts) >= 2 else None for f in rel_files]
        splits = [f.parents[1].name if len(f.parts) >= 3 else None for f in rel_files]
        xtra = pd.DataFrame(dict(path=paths, label=labels, split=splits))
    else:
        xtra = get_xtra(rel_files)
        if len(xtra) != len(files): raise ValueError('get_xtra must return one row per file')
        xtra = pd.concat([pd.DataFrame(dict(path=paths)), xtra.reset_index(drop=True)], axis=1)
    return files, xtra

A small temporary folder mirrors the `split/label/file` layout. `get_folder_data` extracts paths, labels and splits in sorted order, or delegates extra columns to `get_xtra`.

In [36]:
@contextmanager
def sample_folder():
    with tempfile.TemporaryDirectory() as d:
        folder = Path(d)/'data'
        for split in ('train', 'valid'):
            for label, text in [('pos', 'Great film'), ('neg', 'Awful film')]:
                review = folder/split/label/'review.txt'
                review.parent.mkdir(parents=True)
                review.write_text(text)
        yield folder, Path(d)/'cache'

In [37]:
with sample_folder() as (folder, _):
    files, folder_xtra = get_folder_data(folder)
    test_eq(folder_xtra.split.tolist(), ['train', 'train', 'valid', 'valid'])
    test_eq(folder_xtra.label.tolist(), ['neg', 'pos', 'neg', 'pos'])
    _, custom_xtra = get_folder_data(folder, get_xtra=lambda rs: pd.DataFrame(dict(stem=[r.stem for r in rs])))
    test_eq(custom_xtra.columns.tolist(), ['path', 'stem'])
    with expect_fail(ValueError, contains='one row per file'): get_folder_data(folder, get_xtra=lambda rs: pd.DataFrame(dict(a=[1])))
folder_xtra

,path,label,split
0,train/neg/review.txt,neg,train
1,train/pos/review.txt,pos,train
2,valid/neg/review.txt,neg,valid
3,valid/pos/review.txt,pos,valid


In [38]:
@delegates(tokenize_batches, but=['get_texts'])
def tokenize_folder(
    path, # Folder of text files
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    get_xtra=None, # Callable returning `xtra` columns for nonstandard folder layouts
    cache_dir=None, # Cache directory; defaults to a path keyed by the folder's full path and tokenizer
    **kwargs
):
    "Cache the text files under `path` in one Arrow file, with their paths and `xtra` columns"
    path = Path(path)
    files, xtra = get_folder_data(path, get_xtra)
    if cache_dir is None: cache_dir = fastai_path('data')/'token_cache'/str(path.resolve()).lstrip('/')/tok
    cache = Path(cache_dir)/'files.arrow'
    if cache.exists():
        docs, tokenizer, cached = load_token_cache(cache)
        if cached['path'].tolist() == xtra['path'].tolist(): return files, docs, tokenizer, xtra
    tokenizer = hf_tokenizer(tok)
    ids = tokenize_batches(files, tokenizer, get_texts=lambda fs: [f.read_text(encoding='utf-8') for f in fs], **kwargs)
    save_token_cache(cache, ids, tokenizer, xtra)
    return files, load_token_cache(cache)[0], tokenizer, xtra


`tokenize_folder` writes `files.arrow` once and reuses it while the sorted file paths match.

In [39]:
with sample_folder() as (folder, cache_dir):
    files, fdocs, ftok, fxtra = tokenize_folder(folder, cache_dir=cache_dir)
    test_eq((cache_dir/'files.arrow').exists(), True)
    test_eq([ftok.decode(d.tolist()) for d in fdocs], [f.read_text() for f in files])
    _, fdocs2, _, _ = tokenize_folder(folder, cache_dir=cache_dir)
    test_eq(list(fdocs2), list(fdocs))
fxtra

 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

,path,label,split
0,train/neg/review.txt,neg,train
1,train/pos/review.txt,pos,train
2,valid/neg/review.txt,neg,valid
3,valid/pos/review.txt,pos,valid


On the full IMDb dataset, `tokenize_folder` caches all reviews in one Arrow file and returns matching files, documents, tokenizer and `xtra` columns.

In [40]:
#| eval: false
imdb_path = untar_data(URLs.IMDB)
files, imdb_docs, tok_imdb, imdb_xtra = tokenize_folder(imdb_path)
len(files), files[0].relative_to(imdb_path), imdb_xtra.iloc[0], tok_imdb.decode(imdb_docs[0][:30].tolist())

(100002,
 Path('test/neg/0_2.txt'),
 path     test/neg/0_2.txt
 label                 neg
 split                test
 Name: 0, dtype: str,
 'Once again Mr. Costner has dragged out a movie for far longer than necessary. Aside from the terrific sea rescue sequences, of which there are very')

## Transforms

Cached token IDs still need boundary tokens and a tensor type for training. `HfTokenizer` wraps each document in a `TensorText` with `<bos>` and `<eos>`, leaving the cached arrays unchanged. It also accepts raw strings.

In [41]:
class HfTokenizer(Transform):
    "Encode texts or cached token IDs with `tok`, adding `BOS_TOK` and `EOS_TOK`, and decode them back to text"
    input_types = (str, np.ndarray)
    def __init__(self, tok):
        self.tok = tok
        self.bos,self.eos,self.pad_id = (tok.token_to_id(t) for t in (BOS_TOK, EOS_TOK, PAD_TOK))
        self.vocab = [tok.id_to_token(i) for i in range(tok.get_vocab_size())]
    def _add_boundaries(self, ids): return TensorText([self.bos] + ids + [self.eos])
    def _len(self, o): return len(o) if isinstance(o, np.ndarray) else len(self.tok.encode(o).ids)
    def get_lengths(self, items):
        if isinstance(items, TokenDocs): return (items.lens+2)[items.idxs]
        items = L(items)
        if not items: return L()
        return L(self._len(o)+2 for o in items) if isinstance(items[0], (np.ndarray, str)) else None
    def encodes(self, s:str): return self._add_boundaries(self.tok.encode(s).ids)
    def encodes(self, s:np.ndarray): return self._add_boundaries(s.tolist())
    def decodes(self, t:TensorText): return TitledStr(self.tok.decode(t.tolist(), skip_special_tokens=False))

The boundaries add two tokens per document. `get_lengths` reports these lengths to `LMDataLoader` and `SortedDL`.

In [42]:
tfm = HfTokenizer(tok)
t = tfm(docs[0])
test_eq(tfm.get_lengths(texts), tfm.get_lengths(docs))
test_eq(tfm.get_lengths(docs[[1, 0]]), tfm.get_lengths(texts)[[1, 0]])
test_eq(tfm.get_lengths([]), [])
t, tfm.decode(t), tfm.get_lengths(docs)

(TensorText([50258,   464,  2068,  7586, 21831, 50257]),
 '<bos>The quick brown fox<eos>',
 array([6, 8]))

A raw string and its cached IDs produce the same `TensorText`. Decoding keeps the boundary tokens visible.

In [43]:
test_eq(tfm(texts[0]), t)
tfm(texts[1]), tfm.decode(tfm(texts[1]))

(TensorText([50258,    73,  8142,   625,   262, 16931,  3290, 50257]),
 '<bos>jumps over the lazy dog<eos>')

Place `Reverse` after `HfTokenizer` to train on reversed sequences. It reverses on decoding too, restoring the original text for display. The same loaders and learners can use either direction.

In [44]:
class Reverse(Transform):
    "Reverse a token sequence on encode and restore its original order on decode"
    def encodes(self, o): return o.flip(0)
    def decodes(self, o): return o.flip(0)


In [45]:
pipe = Pipeline([tfm, Reverse()])
test_eq(pipe(docs[0]), t.flip(0))
pipe.decode(pipe(docs[0]))


'<bos>The quick brown fox<eos>'

## DataLoaders

`HfTextBlock` uses `LMDataLoader` for language modeling. For classification, it uses `SortedDL` and pads documents with the tokenizer's `pad_id`.

In [46]:
class HfTextBlock(TransformBlock):
    "A `TransformBlock` for documents that `tok_tfm` encodes straight to token ids"
    def __init__(
        self,
        tok_tfm, # Tokenizing `Transform`, such as `HfTokenizer`
        is_lm=False, # Build a language model loader instead of a classifier loader
        seq_len=72, # Language model sequence length or classifier padding chunk size
        backwards=False # Reverse each tokenized document
    ):
        type_tfms = [tok_tfm] + ([Reverse()] if backwards else [])
        dls_kwargs = {'seq_len': seq_len} if is_lm else {'before_batch': Pad_Chunk(seq_len=seq_len, pad_idx=tok_tfm.pad_id)}
        super().__init__(type_tfms=type_tfms, dl_type=LMDataLoader if is_lm else SortedDL, dls_kwargs=dls_kwargs)

A language model predicts the next token in the stream. In this batch, the targets in `y` are shifted one position ahead of the inputs in `x`.

In [47]:
blk = HfTextBlock(tfm, is_lm=True, seq_len=3)
lm_dls = DataBlock(blocks=blk, splitter=RandomSplitter(0.5, seed=42)).dataloaders(docs, bs=1)
x, y = lm_dls.one_batch()
test_eq(x[:, 1:], y[:, :-1])
x, y

(LMTensorText([[50258,    73,  8142]], device='mps:0'),
 TensorText([[  73, 8142,  625]], device='mps:0'))

### Selecting documents

Source splits describe where documents came from. Loader splits select document positions for training and validation. `get_splits` uses the saved split names, or `valid_pct` to make a new random split from all selected rows.

In [48]:
def get_splits(
    xtra, # Cached `xtra` columns, one row per document
    train='train', # Training split name
    valid='validation', # Validation split name
    valid_pct=None, # Random validation fraction; None uses named source splits
    seed=None, # Random split seed
    folders=None # Top-level source folders to include; valid_pct splits all their rows
):
    "Training and validation row positions from cached `xtra` columns"
    rows = L(range_of(xtra))
    if folders is not None:
        folders = L(folders)
        rows = L(mask2idxs(xtra['path'].map(lambda p: Path(p).parts[0] in folders)))
        xtra = xtra.iloc[rows]
    if valid_pct is not None:
        train_idx, valid_idx = RandomSplitter(valid_pct, seed=seed)(rows)
        return rows[train_idx], rows[valid_idx]
    return [rows[mask2idxs(xtra['split'] == name)] for name in (train, valid)]

In [49]:
split_xtra = pd.DataFrame(dict(split=['train', 'train', 'validation', 'test']))
split_indices = get_splits(split_xtra)
test_eq(split_indices, ([0, 1], [2]))
test_eq(get_splits(split_xtra.iloc[:0]), ([], []))
split_indices

[[0, 1], [2]]

Passing `folders` restricts selection to matching top-level directories, and `valid_pct` draws a random split from those selected rows.

In [50]:
folder_split_xtra = pd.DataFrame(dict(path=['train/a.txt', 'train/b.txt', 'unsup/c.txt', 'test/d.txt'], split=['train', 'train', 'unsup', 'test']))
tr, va = get_splits(folder_split_xtra, folders=['train', 'unsup'], valid_pct=0.34, seed=42)
test_eq(sorted(tr + va), [0, 1, 2])
tr, va

([2, 1], [0])

### Loader factories

`TextDataLoadersNew` is the replacement under development for `TextDataLoaders`. `from_docs` builds loaders from `TokenDocs`, with labels supplied separately. `from_dataset` and `from_folder` obtain those documents and their `xtra` columns through their token caches. Both factories accept the loader options of `from_docs`.

In [51]:
class TextDataLoadersNew(DataLoaders):
    "Wrapper around several `DataLoader`s with factory methods for texts encoded by `HfTokenizer`"
    @classmethod
    @delegates(Datasets.dataloaders)
    def from_docs(
        cls,
        docs, # `TokenDocs` to load
        tok, # Tokenizer used to encode the documents
        splits, # Training and validation document indices
        labels=None, # One label per document; None for a language model
        seq_len=72, # Language model sequence length or classifier padding chunk size
        backwards=False, # Reverse each tokenized document
        bs=64, # DataLoader batch size
        **kwargs
    ):
        "Build language model or classification loaders from tokenized documents"
        is_lm = labels is None
        text = HfTextBlock(HfTokenizer(tok), is_lm=is_lm, seq_len=seq_len, backwards=backwards)
        tls = [TfmdLists(docs, text.type_tfms, splits=splits)]
        if not is_lm: tls.append(TfmdLists(labels, CategoryBlock().type_tfms, splits=splits))
        dsets = Datasets(tls=tls, dl_type=text.dl_type)
        kwargs = {**text.dls_kwargs, 'after_item': text.item_tfms, 'after_batch': text.batch_tfms, **kwargs}
        return dsets.dataloaders(bs=bs, **kwargs)

For quick experiments, use `tokenize_texts` and `TokenDocs` to prepare documents in memory, then build loaders with `from_docs`. No cache files are written. Pass `labels` for classification or omit them for language modeling.

In [52]:
memory_dls = TextDataLoadersNew.from_docs(docs, tok, splits=([0], [1]), bs=1, seq_len=3)
test_is(memory_dls.train_ds.items.chunks, docs.chunks)
memory_dls.one_batch()


(LMTensorText([[50258,   464,  2068]], device='mps:0'),
 TensorText([[ 464, 2068, 7586]], device='mps:0'))

### Dataset loaders

In [53]:
@patch(cls_method=True)
@delegates(TextDataLoadersNew.from_docs, but=['labels'])
def from_dataset(
    cls:TextDataLoadersNew,
    name, # Hub dataset name or local dataset directory
    train='train', # Source split used for training
    valid='validation', # Source split used for validation
    valid_pct=None, # Random validation fraction of selected rows; None uses source splits
    seed=None, # Random split seed
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    cache_dir=None, # Token cache directory
    label_col=None, # Label column in the cached `xtra`; None for a language model
    xtra_cols=(), # Additional source columns to cache; `label_col` is included automatically
    subset=None, # Dataset configuration name
    text_col=None, # Text column; required when `get_data` is not supplied
    get_data=None, # Callable returning texts and `xtra` for each source split
    **kwargs
):
    "Build language model or classification loaders using cached `xtra` columns"
    xtra_cols = (L(xtra_cols) + L(label_col)).unique()
    split = [train, valid] if valid_pct is None else None
    docs, tok, xtra = tokenize_dataset(name, subset=subset, split=split, tok=tok,
        cache_dir=cache_dir, text_col=text_col, get_data=get_data, xtra_cols=xtra_cols)
    splits = get_splits(xtra, train=train, valid=valid, valid_pct=valid_pct, seed=seed)
    labels = None if label_col is None else xtra[label_col]
    return cls.from_docs(docs, tok, splits, labels=labels, **kwargs)

`from_dataset` builds a language model unless `label_col` is supplied. For classification, it includes that column in the cached `xtra` automatically; a custom `get_data` callable must provide it.



In [54]:
with sample_dataset() as (source, cache_dir):
    review_dls = TextDataLoadersNew.from_dataset(source, text_col='text', label_col='label',
        cache_dir=cache_dir, bs=2, val_bs=1, device='cpu')
    _, tokenizer, _ = load_token_cache(cache_dir/'train.arrow')
    train_items = [(tokenizer.decode(doc.tolist()), review_dls.vocab[1][label]) for doc, label in review_dls.train_ds]
    valid_items = [(tokenizer.decode(doc.tolist()), review_dls.vocab[1][label]) for doc, label in review_dls.valid_ds]
    review_vocab, review_batch = review_dls.vocab[1], review_dls.one_batch()
    test_is(review_dls.train_ds.items.chunks, review_dls.valid_ds.items.chunks)
test_eq(train_items, [('Great train film', 'pos'), ('Awful train film', 'neg')])
test_eq(valid_items, [('Great validation film', 'pos'), ('Awful validation film', 'neg')])
train_items, valid_items


Generating train split: 2 examples [00:00, 2299.51 examples/s]
Generating validation split: 2 examples [00:00, 4000.29 examples/s]
Generating test split: 2 examples [00:00, 4576.44 examples/s]


 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

([('Great train film', 'pos'), ('Awful train film', 'neg')],
 [('Great validation film', 'pos'), ('Awful validation film', 'neg')])

For classification, `x` contains padded token sequences and `y` contains category indices. The vocabulary maps those indices back to labels.

In [55]:
x, y = review_batch
test_eq(review_vocab, ['neg', 'pos'])
review_vocab, x, y

(['neg', 'pos'],
 TensorText([[50258, 23155,   913,  4512,  2646, 50257],
             [50258, 13681,  4512,  2646, 50257, 50259]]),
 TensorCategory([0, 1]))

With `valid_pct`, `from_dataset` pools every source split before making a new random split. This includes unsupervised documents and any test data. Here all six local reviews participate, including the two from the test source.

In [56]:
with sample_dataset() as (source, cache_dir):
    all_dls = TextDataLoadersNew.from_dataset(source, text_col='text', valid_pct=0.2,
        seed=42, cache_dir=cache_dir, bs=1, seq_len=3)
    source_counts = tokenize_dataset(source, text_col='text', cache_dir=cache_dir)[2]['split'].value_counts()
test_eq(len(all_dls.train_ds) + len(all_dls.valid_ds), source_counts.sum())
dict(sources=source_counts.to_dict(), train=len(all_dls.train_ds), valid=len(all_dls.valid_ds))


Generating train split: 2 examples [00:00, 2267.80 examples/s]
Generating validation split: 2 examples [00:00, 3040.45 examples/s]
Generating test split: 2 examples [00:00, 2624.72 examples/s]


 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

{'sources': {'train': 2, 'validation': 2, 'test': 2}, 'train': 5, 'valid': 1}

For language modeling, leave `label_col` unset. This WikiText example uses the article preparation callable from earlier. `train='train[:1%]'` selects a slice, with that exact expression recorded as its split name. The other source is the named validation split.

In [57]:
#| eval: false
wiki_dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1', get_data=get_wiki103_data, train='train[:1%]', bs=8)
wiki_dls.show_batch(max_n=2)

<bos> = Chris Turner ( American football ) = 
 Chris Turner ( born September 8 , 1987 ) is an American football quarterback . He played quarterback for the Maryland Terrapins at the University of Maryland from 2007 to 2009 . Turner began his career at Maryland as a redshirt in 2005 and then served as a reserve quarterback before he earned the starting position . 
 Darden was drafted in the first round ( 18th overall ) in the 1972 NFL Draft by the Cleveland Browns . Darden started at free safety from his rookie season of 1972 until he lost the job in his tenth and final season to Clinton Burrell . He was selected an All @-@ Pro safety in 1976 , 1978 , and 1979 , and went to
 = Chris Turner ( American football ) = 
 Chris Turner ( born September 8 , 1987 ) is an American football quarterback . He played quarterback for the Maryland Terrapins at the University of Maryland from 2007 to 2009 . Turner began his career at Maryland as a redshirt in 2005 and then served as a reserve quarterback

For classification, use named labeled sources. Here `valid='test'` uses IMDb's test reviews for validation; other source splits are left out.

In [58]:
#| eval: false
imdb_dls = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label', valid='test', bs=8)
imdb_dls.show_batch(max_n=2)


,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",1
1,"<bos>*****WARNING, MAY CONTAIN SPOILERS WHICH WILL BE MORE ENTERTAINING THAN THIS TRIPE.**** <br /><br />Heres some good advise to anyone living in the U.K. Whenever Channel 5 has an old 80's comedy on late at night, read a book instead. I am currently in the process of recovering from a seizure, due to reading some of the comments on this film on here. I am actually shocked at the fact that someone actually said this film was realistic! All I can say is thank god the Cold War never escalated or else we might as well have given the Commie's our borders... I found this film dire in the utmost pretence, maybe it is just my British perception of what makes a film funny, who knows? But in all aspects, this film is not just awful, its teeth grindingly terrible.<br /><br />I've never been a fan of Bill Murray, and",0


### Folder loaders

`from_folder` caches text files under `path` and reads splits and labels from the cached `xtra` columns. Set `label_col` for another label column or pass `get_xtra` for a custom layout.


In [59]:
@patch(cls_method=True)
@delegates(TextDataLoadersNew.from_docs, but=['labels'])
def from_folder(
    cls:TextDataLoadersNew,
    path, # Folder of text files
    train='train', # Training split name in the cached `split` column
    valid='valid', # Validation split name in the cached `split` column
    valid_pct=None, # Random validation fraction; None uses source splits
    seed=None, # Random split seed
    is_lm=False, # Build a language model instead of a classifier
    tok='gpt2', # Tokenizer name for `hf_tokenizer`
    folders=None, # Source folders to include; with `valid_pct`, combine their rows before splitting
    cache_dir=None, # Token cache directory
    label_col='label', # Cached label column for classification
    get_xtra=None, # Callable returning `xtra` columns for nonstandard folder layouts
    **kwargs
):
    "Build language model or classification loaders from a folder's cached `xtra` columns"
    _, docs, tok, xtra = tokenize_folder(path, tok, get_xtra=get_xtra, cache_dir=cache_dir)
    splits = get_splits(xtra, train=train, valid=valid, valid_pct=valid_pct, seed=seed, folders=folders)
    labels = None if is_lm else xtra[label_col]
    return cls.from_docs(docs, tok, splits, labels=labels, path=path, **kwargs)

A small local folder uses the same `split/label/file` layout as IMDb. Both loader types read their documents from the same memory-mapped token arrays.


In [60]:
with sample_folder() as (folder, cache_dir):
    folder_dls = TextDataLoadersNew.from_folder(folder, cache_dir=cache_dir, bs=2, device='cpu')
    folder_lm = TextDataLoadersNew.from_folder(folder, cache_dir=cache_dir, is_lm=True, backwards=True, bs=1, seq_len=3, device='cpu')
    test_eq([folder_dls.vocab[1][label] for _, label in folder_dls.valid_ds], ['neg', 'pos'])
    test_is(folder_dls.train_ds.items.chunks, folder_dls.valid_ds.items.chunks)
    test_eq(folder_lm.train_ds[0][0].flip(0), folder_dls.train_ds[0][0])
    folder_batch, folder_lm_batch = folder_dls.one_batch(), folder_lm.one_batch()
folder_batch, folder_lm_batch

 |███████████████████████████████| 100.00% [1/1 00:00<00:00]

((TensorText([[50258, 23155,   913,  2646, 50257],
              [50258, 13681,  2646, 50257, 50259]]),
  TensorCategory([0, 1])),
 (LMTensorText([[50257,  2646, 13681]]), TensorText([[ 2646, 13681, 50258]])))

For IMDb, the `train` and `test` folders provide split names and `pos` and `neg` provide labels. Here `valid='test'` uses the test reviews for validation.

In [61]:
#| eval: false
imdb_path = untar_data(URLs.IMDB)
imdb_folder_dls = TextDataLoadersNew.from_folder(imdb_path, valid='test', bs=8)
imdb_folder_dls.show_batch(max_n=2)

,text,category
0,"<bos>Match 1: Tag Team Table Match Bubba Ray and Spike Dudley vs Eddie Guerrero and Chris Benoit Bubba Ray and Spike Dudley started things off with a Tag Team Table Match against Eddie Guerrero and Chris Benoit. According to the rules of the match, both opponents have to go through tables in order to get the win. Benoit and Guerrero heated up early on by taking turns hammering first Spike and then Bubba Ray. A German suplex by Benoit to Bubba took the wind out of the Dudley brother. Spike tried to help his brother, but the referee restrained him while Benoit and Guerrero ganged up on him in the corner. With Benoit stomping away on Bubba, Guerrero set up a table outside. Spike dashed into the ring and somersaulted over the top rope onto Guerrero on the outside! After recovering and taking care of Spike, Guerrero slipped a table",pos
1,"<bos>Peter Jacksons version(s) are better films overall from objective point of view. That being said, they are not my favorite screen versions of Lord Of The Rings, and let me explain why. <br /><br />Firstly, the acting of the on-screen characters is just too ordinary and uninspiring with Jackson's LOTR. The whole cast is too run of the mill. ""Are you claiming that those silly cartoon characters of Ralph Bakshi version are better actors than real people?"" one could ask. Well, they are not really silly(save for Hobbits, later about them) and they certainly pack more personality than Jackson's party - even with much more limited dialogue time. And that is because of superior _voice_ acting of the Bakshi's LOTR. Take Aragorn for example. In this version his voice is deep and charismatic with full of authority(Aragorn the lord)and with a seasoned rasp(Aragorn the ranger). This is due to John",pos


For language modeling, the random split below pools all cached documents, including unsupervised reviews. Pass `folders` to restrict that pool.

In [62]:
#| eval: false
imdb_lm = TextDataLoadersNew.from_folder(imdb_path, is_lm=True, valid_pct=0.1, bs=64)
len(imdb_lm.train_ds), len(imdb_lm.valid_ds), len(imdb_lm.vocab)

(90002, 10000, 50260)

## Export -

In [63]:
#| hide
from nbdev import nbdev_export
nbdev_export()